# Dubai Building Code — FAISS Index Builder
Embeds `dubai_chunks.json` using `nomic-embed-text-v1` (same model as local Ollama) and produces:
- `index.faiss` — FAISS IndexFlatIP (cosine via inner product on L2-normalised vectors)
- `index_meta.pkl` — chunk metadata list

**Runtime:** GPU (A100 recommended). Upload `dubai_chunks.json` when prompted, then download the two output files into `rag_api/`.

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q sentence-transformers faiss-cpu einops

In [ ]:
# ── 2. Upload dubai_chunks.json ───────────────────────────────────────────────
from google.colab import files
print('Upload dubai_chunks.json ...')
uploaded = files.upload()
assert 'dubai_chunks.json' in uploaded, 'Please upload dubai_chunks.json'
print('Uploaded OK')

In [ ]:
# ── 3. Load chunks ────────────────────────────────────────────────────────────
import json
with open('dubai_chunks.json', encoding='utf-8') as f:
    chunks = json.load(f)

print(f'Chunks loaded : {len(chunks)}')
print(f'Sample text   : {chunks[10]["text"][:120]}...')

In [ ]:
# ── 4. Load nomic-embed-text-v1 ───────────────────────────────────────────────
# Same model that Ollama serves as 'nomic-embed-text' — vectors are compatible.
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

model = SentenceTransformer(
    'nomic-ai/nomic-embed-text-v1',
    trust_remote_code=True,
    device=device,
)
print('Model loaded')

In [ ]:
# ── 5. Embed all chunks ───────────────────────────────────────────────────────
# Prefix must match build_index.py: 'search_document: {text}'
import numpy as np
from tqdm.notebook import tqdm

BATCH_SIZE = 64

texts = [f"search_document: {c['text']}" for c in chunks]

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2-normalise for cosine via inner product
    device=device,
)

embeddings = np.array(embeddings, dtype='float32')
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# ── 6. Build FAISS index ──────────────────────────────────────────────────────
import faiss

dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)   # inner product on normalised vecs = cosine
index.add(embeddings)

faiss.write_index(index, 'index.faiss')
print(f'FAISS index: {index.ntotal} vectors, dim={dim}')

In [ ]:
# ── 7. Save metadata ──────────────────────────────────────────────────────────
import pickle

meta = [
    {
        'id':         c['id'],
        'section':    c.get('section'),
        'page_start': c.get('page_start'),
        'page_end':   c.get('page_end'),
        'text':       c['text'],
    }
    for c in chunks
]

with open('index_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print(f'Metadata saved: {len(meta)} entries')

In [ ]:
# ── 8. Quick sanity check ─────────────────────────────────────────────────────
query = 'search_query: what is the minimum corridor width'
q_vec = model.encode([query], normalize_embeddings=True)
scores, indices = index.search(np.array(q_vec, dtype='float32'), k=3)

print('Top 3 results for:', query)
for score, idx in zip(scores[0], indices[0]):
    c = meta[idx]
    print(f'  score={score:.3f} | p{c["page_start"]}-{c["page_end"]} | {c["section"]}')    
    print(f'  {c["text"][:150]}...')
    print()

In [ ]:
# ── 9. Download outputs ───────────────────────────────────────────────────────
# Copy both files into rag_api/ on your local machine
files.download('index.faiss')
files.download('index_meta.pkl')